# Train DDPM on CIFAR-10 (Google Colab)

This notebook is a **thin runner** — all training logic lives in `src/`.

**Steps:**
1. Enable GPU runtime (Runtime → Change runtime type → GPU)
2. Clone / mount the repository
3. Install requirements
4. Verify CUDA
5. Import `Trainer` and start training
6. Plot loss history
7. List saved checkpoints

Do **not** duplicate the training loop here.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (recommended)

Checkpoints saved only under `/content` disappear when Colab disconnects.
Mount Drive and use it as `backup_dir` so trained models are kept.

In [ ]:
# Clone repo (change URL to your repo)
!git clone https://github.com/Temsegn/diffusion-model-from-scratch.git
%cd diffusion-model-from-scratch

# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

PROJECT = Path.cwd()
if not (PROJECT / "src").exists():
    raise FileNotFoundError("Run this notebook from the project root (folder with src/)")

print("Project root:", PROJECT.resolve())
print("Drive mounted at /content/drive")

## 3. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 4. Verify CUDA

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 5. Import Trainer and start training

Default: 100 epochs, batch size 128, AdamW lr=2e-4, T=1000.
For a quick smoke test, set `cfg.epochs = 1`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import TrainConfig
from src.training.trainer import Trainer

cfg = TrainConfig()

# Quick smoke test:
# cfg.epochs = 1

# Save checkpoints locally during training
cfg.checkpoint_dir = "./checkpoints"

# Persistent backup on Google Drive (survives Colab disconnect)
cfg.backup_dir = "/content/drive/MyDrive/ddpm-checkpoints"

# Optional: also cache CIFAR-10 on Drive (skip re-download next session)
cfg.data_root = "/content/drive/MyDrive/ddpm-data"

# Save checkpoints at these epochs
cfg.save_epochs = [10, 50, 100]

trainer = Trainer(cfg)
loss_history = trainer.train()

## 6. Plot training loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(trainer.loss_history) + 1), trainer.loss_history)
plt.xlabel("Epoch")
plt.ylabel("Average MSE loss")
plt.title("DDPM training loss (noise prediction)")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Saved checkpoint locations

In [ ]:
from pathlib import Path

ckpt_dir = Path(cfg.checkpoint_dir)
backup_dir = Path(cfg.backup_dir) if cfg.backup_dir else None

print(f"Local checkpoints: {ckpt_dir.resolve()}")
for p in sorted(ckpt_dir.glob("ddpm_epoch_*.pt")):
    print(" -", p.name, f"({p.stat().st_size / 1e6:.1f} MB)")

if backup_dir and backup_dir.exists():
    print(f"\nDrive backup: {backup_dir.resolve()}")
    for p in sorted(backup_dir.glob("ddpm_epoch_*.pt")):
        print(" -", p.name, f"({p.stat().st_size / 1e6:.1f} MB)")
    loss_file = backup_dir / "loss_history.json"
    if loss_file.exists():
        print(" - loss_history.json")